<a href="https://colab.research.google.com/github/trialdyk/marchine-learning/blob/main/JS09/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Pembelajaran Mesin - JS09 Approximate Nearest Neighbors (ANN)
## Tugas Lab 2: Exact NN, ANNOY, FAISS, dan HNSW pada Dataset Spotify

Dataset: Spotify Songs with Attributes and Lyrics (file pertama, `songs_with_attributes_and_lyrics.csv`).

### Langkah 1 - Instal Library dan Unduh Dataset

In [1]:
!pip install annoy faiss-cpu hnswlib
import os
if not os.path.exists('songs_with_attributes_and_lyrics.csv'):
    !curl -s -L -A "Mozilla/5.0" -o spotify.zip "https://www.kaggle.com/api/v1/datasets/download/bwandowando/spotify-songs-with-attributes-and-lyrics"
    !unzip -q -o spotify.zip songs_with_attributes_and_lyrics.csv

### Langkah 2 - Load Dataset dan Standarisasi Fitur

In [2]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
df = pd.read_csv('songs_with_attributes_and_lyrics.csv', usecols=['id', 'name', 'artists'] + features)
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10
dim = X_scaled.shape[1]
print(df.shape)
df.head()

(955320, 12)


,id,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0Prct5TDjAnEgIqbxcldY9,!,HELLYEAH,0.415,0.6050,-11.157,0.0575,0.00116,0.838000,0.4710,0.193,100.059
1,2ASl4wirkeYm3OWZxXKYuq,!!,Yxngxr1,0.788,0.6480,-9.135,0.3150,0.90000,0.000000,0.1760,0.287,79.998
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Glowie,0.000,0.0354,-20.151,0.0000,0.90800,0.000000,0.4790,0.000,0.000
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Rosendo,0.657,0.8820,-6.340,0.0385,0.00740,0.000013,0.0474,0.939,123.588
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Rosendo,0.659,0.8930,-8.531,0.0411,0.09220,0.000019,0.0534,0.951,123.600


### Langkah 3 - Bangun Index

In [3]:
t = time.time()
exact = NearestNeighbors(n_neighbors=k + 1, algorithm='brute', metric='euclidean')
exact.fit(X_scaled)
build_exact = time.time() - t

t = time.time()
annoy_index = AnnoyIndex(dim, 'euclidean')
for i in range(len(X_scaled)):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(10)
build_annoy = time.time() - t

t = time.time()
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(X_scaled)
build_faiss = time.time() - t

t = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=dim)
hnsw_index.init_index(max_elements=len(X_scaled), ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50)
build_hnsw = time.time() - t

pd.DataFrame({'Algoritma': ['Exact NN', 'ANNOY', 'FAISS', 'HNSW'],
              'Build (detik)': [build_exact, build_annoy, build_faiss, build_hnsw]})

,Algoritma,Build (detik)
0,Exact NN,0.023938
1,ANNOY,9.432134
2,FAISS,0.094702
3,HNSW,98.446809


### Langkah 4 - Pencarian Track Terdekat

In [4]:
def search_exact(q):
    return exact.kneighbors(q.reshape(1, -1), n_neighbors=k + 1, return_distance=False)[0]

def search_annoy(q):
    return np.array(annoy_index.get_nns_by_vector(q, k + 1))

def search_faiss(q):
    return faiss_index.search(q.reshape(1, -1), k + 1)[1][0]

def search_hnsw(q):
    return hnsw_index.knn_query(q.reshape(1, -1), k=k + 1)[0][0]

searchers = {'Exact NN': search_exact, 'ANNOY': search_annoy, 'FAISS': search_faiss, 'HNSW': search_hnsw}

query_idx = 1000
q = X_scaled[query_idx]
print('Track query:', df.loc[query_idx, 'name'], '-', df.loc[query_idx, 'artists'])

hasil_query = {}
for nama, fn in searchers.items():
    t = time.time()
    idx = fn(q)
    waktu = time.time() - t
    idx = [i for i in idx if i != query_idx][:k]
    hasil_query[nama] = idx
    print(f'\n=== {nama} (query {waktu * 1000:.3f} ms) ===')
    print(df.loc[idx, ['name', 'artists']].to_string())

Track query: (Come 'Round Here) I'm The One You Need - Smokey Robinson & The Miracles

=== Exact NN (query 104.292 ms) ===
                                                                          name                             artists
434574                                              La Vida del Leon - En Vivo                       Alta Consigna
190385                                                    Dia Branco - Ao Vivo  Alceu Valença;Orquestra Ouro Preto
878270                                               What the Hell Is Goin' on          Elvin Bishop, Kid Andersen
625534                                                  Quand le rêve est fini  Plume Latraverse anf The Plumettes
617507                                                         Pretty Flamingo                        Manfred Mann
617510                                                         Pretty Flamingo                        Manfred Mann
141347                                Cien Kilos de Barro (100 Pounds of

                                                                          name                             artists
434574                                              La Vida del Leon - En Vivo                       Alta Consigna
190385                                                    Dia Branco - Ao Vivo  Alceu Valença;Orquestra Ouro Preto
878270                                               What the Hell Is Goin' on          Elvin Bishop, Kid Andersen
625534                                                  Quand le rêve est fini  Plume Latraverse anf The Plumettes
617507                                                         Pretty Flamingo                        Manfred Mann
617510                                                         Pretty Flamingo                        Manfred Mann
141347                                Cien Kilos de Barro (100 Pounds of Clay)                      Enrique Guzman
238003                                                   Estoy Enamorado de Ti  

### Langkah 5 - Perbandingan Waktu dan Recall (100 Query Acak)

In [5]:
rng = np.random.default_rng(42)
query_ids = rng.choice(len(X_scaled), 100, replace=False)

truth = {}
for qi in query_ids:
    truth[qi] = set(i for i in search_exact(X_scaled[qi]) if i != qi)

rows = []
for nama, fn in searchers.items():
    t = time.time()
    rec = []
    for qi in query_ids:
        found = set(i for i in fn(X_scaled[qi]) if i != qi)
        rec.append(len(found & truth[qi]) / k)
    rows.append({'Algoritma': nama, 'Query rata-rata (ms)': (time.time() - t) / len(query_ids) * 1000,
                 'Recall@10': np.mean(rec)})

pd.DataFrame(rows)

,Algoritma,Query rata-rata (ms),Recall@10
0,Exact NN,34.475877,1.000
1,ANNOY,0.923162,0.828
2,FAISS,5.504103,0.998
3,HNSW,0.181761,0.995


### Analisis

Dataset terdiri dari 955.320 lagu dengan 9 fitur numerik yang sudah distandarisasi dengan StandardScaler. Pencarian dilakukan untuk 10 track terdekat (jarak Euclidean).

**Waktu build.** Exact NN (0,02 detik) dan FAISS Flat (0,09 detik) hanya menyimpan data, sehingga hampir tanpa biaya. ANNOY membutuhkan 9,4 detik untuk membangun 10 tree, dan HNSW paling lama, 98,4 detik, karena membangun graf berlapis (`M=16`, `ef_construction=200`).

**Waktu query (rata-rata 100 query acak).** Exact NN (scikit-learn brute force) paling lambat, 34,5 ms per query. FAISS Flat 5,5 ms (sekitar 6 kali lebih cepat dari Exact NN karena implementasi C++ yang teroptimasi, meskipun tetap memindai semua data). ANNOY 0,92 ms (sekitar 37 kali lebih cepat) dan HNSW 0,18 ms (sekitar 190 kali lebih cepat).

**Akurasi (Recall@10 terhadap Exact NN).** FAISS 0,998 dan HNSW 0,995 hampir sama dengan hasil exact. ANNOY 0,828, artinya rata-rata sekitar 8 dari 10 tetangga sama dengan hasil exact. FAISS Flat seharusnya 1,0; selisih kecil terjadi karena banyak lagu yang memiliki fitur identik atau hampir identik (misalnya lagu duplikat), sehingga urutan pada jarak yang sama bisa berbeda.

**Contoh track terdekat.** Pada query track pertama, hasil ANNOY, FAISS, dan HNSW dapat dibandingkan langsung dengan hasil Exact NN pada Langkah 4. Track dengan fitur yang sama persis (mis. dua versi lagu yang sama) muncul bersamaan sebagai tetangga.

**Kesimpulan.** HNSW memberi kombinasi terbaik antara kecepatan query dan akurasi, tetapi dengan waktu build terlama. ANNOY membangun lebih cepat tetapi akurasinya paling rendah dengan 10 tree. FAISS Flat tetap exact dan cukup cepat untuk data sebesar ini.